In [1]:
import os
import pickle
import numpy as np
import pandas as pd

from tqdm import tqdm
from matplotlib import pyplot as plt

import tensorflow as tf
import keras.backend as K

from tensorflow.keras.models import Model, Sequential, load_model
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Activation, Flatten # usually you end on a Dense layer, Flatten before the dense layer
from tensorflow.keras.callbacks import EarlyStopping, History, ModelCheckpoint

# XGBoost
from xgboost import XGBClassifier

# SVM
import joblib
from sklearn.svm import SVC

# XGBoost + SVM
from sklearn.metrics import accuracy_score

In [2]:
def f1_score(y_true, y_pred): #taken from old keras source code
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    precision = true_positives / (predicted_positives + K.epsilon())
    recall = true_positives / (possible_positives + K.epsilon())
    f1_val = 2*(precision*recall)/(precision+recall+K.epsilon())
    return f1_val

DISPOSITIVOS = ['dish washer', 'kettle', 'microwave', 'washing machine', 'fridge']
#DISPOSITIVOS = ['kettle']
IMAGENS = ['rp', 'gadf','gasf','mtf','tbg']
#IMAGENS = ['tbg']

## Divisão dos dados Treino, Validação e Teste (Caso não haja divisão)

In [3]:
def dividir_dados():
    for disp in DISPOSITIVOS:

        y = pickle.load(open(f"pickle_data_tbg/y_({disp}).pickle","rb"))

        y_train = y[:int(len(y)*0.6)]
        y_val = y[len(y_train):len(y_train) + int(len(y)*0.2)]
        y_test = y[(len(y_train) + len(y_val)):]

        pickle_out = open(f"pickle_data_tbg/y_train({disp}).pickle", "wb")
        pickle.dump(y_train, pickle_out)
        pickle_out.close()

        pickle_out = open(f"pickle_data_tbg/y_val({disp}).pickle", "wb")
        pickle.dump(y_val, pickle_out)
        pickle_out.close()

        pickle_out = open(f"pickle_data_tbg/y_test({disp}).pickle", "wb")
        pickle.dump(y_test, pickle_out)
        pickle_out.close()

        for img in IMAGENS:

            X = pickle.load(open(f"pickle_data_tbg/X_{img}_({disp}).pickle","rb"))

            X_train = X[:int(len(X)*0.6)]
            X_val = X[len(X_train):len(X_train) + int(len(X)*0.2)]
            X_test = X[(len(X_train) + len(X_val)):]

            pickle_out = open(f"pickle_data_tbg/X_{img}_train({disp}).pickle", "wb")
            pickle.dump(X_train, pickle_out)
            pickle_out.close()

            pickle_out = open(f"pickle_data_tbg/X_{img}_val({disp}).pickle", "wb")
            pickle.dump(X_val, pickle_out)
            pickle_out.close()

            pickle_out = open(f"pickle_data_tbg/X_{img}_test({disp}).pickle", "wb")
            pickle.dump(X_test, pickle_out)
            pickle_out.close()

    print("Divisão concluida com sucesso.")

def dividir_dados_val_only():
    for disp in DISPOSITIVOS:

        y = pickle.load(open(f"pickle_data/y_train({disp}).pickle","rb"))

        y_train = y[:int(len(y)*0.8)]
        y_val = y[len(y_train):]

        pickle_out = open(f"pickle_data/y_train_split({disp}).pickle", "wb")
        pickle.dump(y_train, pickle_out)
        pickle_out.close()

        pickle_out = open(f"pickle_data/y_val({disp}).pickle", "wb")
        pickle.dump(y_val, pickle_out)
        pickle_out.close()

        for img in IMAGENS:

            X = pickle.load(open(f"pickle_data/X_{img}_train({disp}).pickle","rb"))

            X_train = X[:int(len(X)*0.8)]
            X_val = X[len(X_train):]

            pickle_out = open(f"pickle_data/X_{img}_train_split({disp}).pickle", "wb")
            pickle.dump(X_train, pickle_out)
            pickle_out.close()

            pickle_out = open(f"pickle_data/X_{img}_val({disp}).pickle", "wb")
            pickle.dump(X_val, pickle_out)
            pickle_out.close()

    print("Divisão concluida com sucesso.")

#dividir_dados_val_only()

## Extração de Features, Transfer Learning c/ MobileNet

In [4]:
# Gerando os outputs de extração de características da MobileNet (checkpoint)
# Somente rodar na primeira vez.
# Para evitar rodar novamente, mantendo "FE_run = False"

batch = 32
folder_i = 'pickle_data'
folder_o = 'FE_output'

FE_run = False
if FE_run:
    for disp in DISPOSITIVOS:
        for img in IMAGENS:

            # Carregamento dos dados
            X_train = pickle.load(open(f"{folder_i}/X_{img}_train({disp}).pickle","rb"))
            X_val = pickle.load(open(f"{folder_i}/X_{img}_val({disp}).pickle","rb"))
            X_test = pickle.load(open(f"{folder_i}/X_{img}_test({disp}).pickle","rb"))

            input_shape = X_train.shape[1:]
            mobilenet = MobileNetV3Large(input_shape = input_shape, weights='imagenet', include_top=False)

            for layer in mobilenet.layers:
                layer.trainable = False

            global_average_pooling_output = GlobalAveragePooling2D()(mobilenet.output)

            FE = Model(inputs=mobilenet.input, outputs=global_average_pooling_output)

            X_train_features = FE.predict(X_train, batch_size=batch, verbose=0)
            X_val_features   = FE.predict(X_val, batch_size=batch, verbose=0)
            X_test_features  = FE.predict(X_test, batch_size=batch, verbose=0)

            with open(f"{folder_o}/XFE_{img}_train({disp}).pickle", 'wb') as f:
                pickle.dump(X_train_features, f)

            with open(f"{folder_o}/XFE_{img}_val({disp}).pickle", 'wb') as f:
                pickle.dump(X_val_features, f)

            with open(f"{folder_o}/XFE_{img}_test({disp}).pickle", 'wb') as f:
                pickle.dump(X_test_features, f)


## Classificação

In [5]:
# Declarando as pastas
folder_X = 'FE_output'
folder_y = 'pickle_data'

# Parâmetros Gerais
model_choice = 'SVM' # 'MLP', 'XGBoost', 'SVM'

# MLP
mlp_n_runs = 5
mlp_fit_params = {
    'batch_size': batch,
    'epochs': 100,
    'verbose': 0
}

mlp_early_stopping_params = {
    'monitor': 'val_loss',
    'mode': 'min',
    'patience': 7,
    'verbose': 0,
    'restore_best_weights': False
}

mlp_model_checkpoint_params = {
    'monitor': 'val_accuracy',
    'mode': 'max',
    'verbose': 0,
    'save_best_only': True
}

# XGBoost
xgb_model_params = {
    'n_estimators': 400,
    'eval_metric': 'logloss',
    'early_stopping_rounds': 7,
}

xgb_fit_params = {
    'verbose': 0
}

# SVM
svm_model_params = {
    'kernel': 'rbf',
    #'probability': True
}

for model_choice in ['SVM', 'XGBoost', 'MLP']:

    df = pd.DataFrame(columns=IMAGENS, index=DISPOSITIVOS)
    for disp in DISPOSITIVOS:
        for img in IMAGENS:
            results_arr = []

            # Carregamento dos dados
            X_train = pickle.load(open(f"{folder_X}/XFE_{img}_train({disp}).pickle","rb"))
            y_train = pickle.load(open(f"{folder_y}/y_train({disp}).pickle","rb"))
            X_val = pickle.load(open(f"{folder_X}/XFE_{img}_val({disp}).pickle","rb"))
            y_val = pickle.load(open(f"{folder_y}/y_val({disp}).pickle","rb"))
            X_test = pickle.load(open(f"{folder_X}/XFE_{img}_test({disp}).pickle","rb"))
            y_test = pickle.load(open(f"{folder_y}/y_test({disp}).pickle","rb"))

            input_shape = X_train.shape[1:]
            if model_choice == 'MLP':
                for run in tqdm(range(mlp_n_runs), desc=f"Executando para {disp}-{img} usando {model_choice}."):

                    mlp_input = tf.keras.layers.Input(shape=input_shape)

                    x = Dense(64, activation='relu')(mlp_input)
                    x = Dropout(0.25)(x)
                    x = Dense(64, activation='relu')(x)
                    x = Dropout(0.25)(x)
                    mlp_output = Dense(1, activation='sigmoid')(x)

                    model = Model(inputs=mlp_input, outputs=mlp_output)

                    model.compile(loss="binary_crossentropy",
                              optimizer="adam",
                              metrics=['accuracy'])

                    checkpoint_filepath = f'ckpt/{disp}/{img}/{model_choice}/{run}/checkpoint.model.keras'
                    os.makedirs(os.path.dirname(checkpoint_filepath), exist_ok=True)
                    history = model.fit(X_train, y_train,
                                        **mlp_fit_params,
                                        callbacks=[EarlyStopping(**mlp_early_stopping_params),
                                                   ModelCheckpoint(checkpoint_filepath,
                                                                   **mlp_model_checkpoint_params)],
                                        validation_data=(X_val,y_val))

                    saved_model = tf.keras.models.load_model(checkpoint_filepath)
                    results = saved_model.evaluate(X_test, y_test, batch_size=batch, verbose=0)
                    results_arr.append(results[1])

            if model_choice == 'XGBoost':
                print(f"Executando para {disp}-{img} usando {model_choice}.")
                model = XGBClassifier(**xgb_model_params)

                model.fit(X_train, y_train,
                          eval_set = [(X_val, y_val)],
                          **xgb_fit_params)

                model_filepath = f'ckpt/{disp}/{img}/{model_choice}/model.json'
                os.makedirs(os.path.dirname(model_filepath), exist_ok=True)
                model.save_model(model_filepath)

                preds = model.predict(X_test)
                acc = accuracy_score(y_test, preds)
                results_arr.append(acc)

            if model_choice == 'SVM':
                print(f"Executando para {disp}-{img} usando {model_choice}.")

                X_train2 = np.vstack((X_train, X_val))
                y_train2 = np.concatenate((y_train, y_val))

                model = SVC(**svm_model_params)
                model.fit(X_train2, y_train2)

                model_filepath = f'ckpt/{disp}/{img}/{model_choice}/model.joblib'
                os.makedirs(os.path.dirname(model_filepath), exist_ok=True)
                joblib.dump(model, model_filepath)

                preds = model.predict(X_test)
                acc = accuracy_score(y_test, preds)
                results_arr.append(acc)

            if model_choice == 'MLP':
                print(f"Acurácia nas runs: {results_arr}")
                print(f"Acurácia Média: {np.mean(np.array(results_arr))}\n")
            else:
                print(f"Acurácia: {results_arr[0]}\n")

            os.makedirs(f'ckpt/{disp}/{img}/{model_choice}', exist_ok=True)
            np.save(f'ckpt/{disp}/{img}/{model_choice}/results_arr.npy', np.array(results_arr)) # Acurácia mèdia da combinação específica
            df.loc[disp, img] = np.mean(np.array(results_arr))

    os.makedirs(f"output/{model_choice}", exist_ok=True)
    df.to_pickle(f"output/{model_choice}/model_eval.pkl") # Acurácia média de cada combinação
df


Executando para dish washer-rp usando SVM.
Acurácia: 0.7662337662337663

Executando para dish washer-gadf usando SVM.
Acurácia: 0.564935064935065

Executando para dish washer-gasf usando SVM.
Acurácia: 0.7792207792207793

Executando para dish washer-mtf usando SVM.
Acurácia: 0.8766233766233766

Executando para dish washer-tbg usando SVM.
Acurácia: 0.8766233766233766

Executando para kettle-rp usando SVM.
Acurácia: 0.9427083333333334

Executando para kettle-gadf usando SVM.
Acurácia: 0.6510416666666666

Executando para kettle-gasf usando SVM.
Acurácia: 0.9583333333333334

Executando para kettle-mtf usando SVM.
Acurácia: 0.9635416666666666

Executando para kettle-tbg usando SVM.
Acurácia: 0.9739583333333334

Executando para microwave-rp usando SVM.
Acurácia: 0.5588235294117647

Executando para microwave-gadf usando SVM.
Acurácia: 0.5686274509803921

Executando para microwave-gasf usando SVM.
Acurácia: 0.7549019607843137

Executando para microwave-mtf usando SVM.
Acurácia: 0.7843137254901

Executando para dish washer-rp usando MLP.: 100%|██████████| 5/5 [00:46<00:00,  9.25s/it]


Acurácia nas runs: [0.8246753215789795, 0.8051947951316833, 0.7922077775001526, 0.7857142686843872, 0.8051947951316833]
Acurácia Média: 0.8025973916053772



Executando para dish washer-gadf usando MLP.: 100%|██████████| 5/5 [00:21<00:00,  4.23s/it]


Acurácia nas runs: [0.7597402334213257, 0.7272727489471436, 0.7727272510528564, 0.6103895902633667, 0.7792207598686218]
Acurácia Média: 0.7298701167106628



...

Executando para fridge-mtf usando MLP.: 100%|██████████| 5/5 [01:17<00:00, 15.53s/it]


Acurácia nas runs: [0.5958254337310791, 0.6252371668815613, 0.6034155488014221, 0.5901328325271606, 0.6271347403526306]
Acurácia Média: 0.6083491444587708



Executando para fridge-tbg usando MLP.: 100%|██████████| 5/5 [01:19<00:00, 15.87s/it]

Acurácia nas runs: [0.5379506349563599, 0.5313093066215515, 0.5388994216918945, 0.5417457222938538, 0.5227704048156738]
Acurácia Média: 0.5345350980758667



,rp,gadf,gasf,mtf,tbg
dish washer,0.802597,0.72987,0.820779,0.874026,0.933766
kettle,0.948958,0.784375,0.96875,0.969792,0.972917
microwave,0.656863,0.658824,0.772549,0.784314,0.915686
washing machine,0.8,0.868966,0.844828,0.758621,0.924138
fridge,0.517078,0.599431,0.595825,0.608349,0.534535


## Versão antiga, com fine-tuning. Não cheguei a usar.

In [ ]:
batch = 32
n_epoch = 100

n_runs = 1

flag_fine_tuning = False
model_choice = 'MLP' # 'MLP', 'XGBoost', 'SVM'

df = pd.DataFrame(columns=IMAGENS, index=DISPOSITIVOS)

for disp in DISPOSITIVOS:
    for img in IMAGENS:
        results_arr = []
        for run in tqdm(range(n_runs), desc=f"Executando para {disp}-{img}"):
            print(f"Run {run+1} de {n_runs}.")
            folder = 'pickle_data'
            X_train = pickle.load(open(f"{folder}/X_{img}_train({disp}).pickle","rb"))
            y_train = pickle.load(open(f"{folder}/y_train({disp}).pickle","rb"))
            X_val = pickle.load(open(f"{folder}/X_{img}_val({disp}).pickle","rb"))
            y_val = pickle.load(open(f"{folder}/y_val({disp}).pickle","rb"))
            X_test = pickle.load(open(f"{folder}/X_{img}_test({disp}).pickle","rb"))
            y_test = pickle.load(open(f"{folder}/y_test({disp}).pickle","rb"))

            input_shape = X_train.shape[1:]
            mobilenet = MobileNetV3Large(input_shape = input_shape, weights='imagenet', include_top=False)
            # Desabilitar Fine-Tuning
            for layer in mobilenet.layers:
                layer.trainable = False

            #x = Flatten()(mobilenet.output)
            x = GlobalAveragePooling2D()(mobilenet.output)
            #if model_choice == 'MLP':
            #x = Dropout(0.25)(x)

            x = Dense(64, activation = 'relu')(x)
            x = Dropout(0.25)(x)

            x = Dense(64, activation = 'relu')(x) #nova
            x = Dropout(0.25)(x) #nova

            x = Dense(1, activation = 'sigmoid')(x)

            model = Model(inputs = mobilenet.input, outputs = x)
            model.compile(loss="binary_crossentropy",
                          optimizer="adam",
                          metrics=['accuracy'])#, f1_score])

            #print(model.summary())

            checkpoint_filepath = f'ckpt/{disp}/{img}/{run}/checkpoint.model.keras'
            history = model.fit(X_train, y_train,
                                batch_size=batch,
                                epochs=n_epoch,
                                verbose=0,
                                callbacks=[EarlyStopping(monitor='val_loss',
                                                         mode='min',
                                                         patience=7,
                                                         verbose=0,
                                                         restore_best_weights=False),
                                           ModelCheckpoint(checkpoint_filepath,
                                                           monitor='val_accuracy',
                                                           mode='max',
                                                           verbose=0,
                                                           save_best_only=True)],
                                validation_data=(X_val,y_val))

            if flag_fine_tuning:    # dado pelo chatgpt, verificar!!!
                fine_tune_at = int(len(mobilenet.layers) * 0.8)

                for layer in mobilenet.layers[:fine_tune_at]:
                    layer.trainable = False
                for layer in mobilenet.layers[fine_tune_at:]:
                    if not isinstance(layer, tf.keras.layers.BatchNormalization):
                        layer.trainable = True

                model.compile(
                    loss="binary_crossentropy",
                    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
                    metrics=['accuracy']
                )

                history_ft = model.fit(
                    X_train, y_train,
                    batch_size=batch,
                    epochs=n_epoch,
                    verbose=0,
                    validation_data=(X_val, y_val),
                    callbacks=[EarlyStopping(monitor='val_loss',
                                                         mode='min',
                                                         patience=5,
                                                         verbose=0,
                                                         restore_best_weights=False),
                                           ModelCheckpoint(checkpoint_filepath,
                                                           monitor='val_accuracy',
                                                           mode='max',
                                                           verbose=0,
                                                           save_best_only=True)]
                )

            saved_model = tf.keras.models.load_model(checkpoint_filepath)
            results = saved_model.evaluate(X_test, y_test, batch_size=batch, verbose=0)
            results_arr.append(results[1])
        print(f"Acurácia nas runs: {results_arr}")
        print(f"Acurácia Média: {np.mean(np.array(results_arr))}\n")
        np.save(f'ckpt/{disp}/{img}/results_arr.npy', np.array(results_arr))
        df.loc[disp, img] = np.mean(np.array(results_arr))

df.to_pickle("output\model_eval.pkl")
df

Executando para dish washer-rp:   0%|          | 0/5 [00:00<?, ?it/s]

Run 1 de 5.


Executando para dish washer-rp:  20%|██        | 1/5 [02:17<09:08, 137.16s/it]

Run 2 de 5.


Executando para dish washer-rp:  40%|████      | 2/5 [04:58<07:34, 151.64s/it]

...

Run 4 de 5.


Executando para fridge-mtf:  80%|████████  | 4/5 [6:03:08<1:16:28, 4588.29s/it]

Run 5 de 5.


Executando para fridge-mtf:  80%|████████  | 4/5 [6:14:14<1:33:33, 5613.55s/it]


KeyboardInterrupt: 